# Step 1: raw dataset generation

In [ ]:
from datasets import Dataset, DatasetDict
import torch
import pickle
import os
from ICL.datasets.RHM import RandomHierarchyModel  # Your existing class


def create_hf_dataset_from_rhm(config_list, samples_per_config=1000, output_dir='./hf_datasets'):
    """Use existing RHM class to generate HuggingFace compatible dataset"""
    
    all_input_ids = []
    all_labels = []  
    all_task_ids = []
    all_config_L = []
    all_config_m = []
    all_lengths = []
    all_rules = {}
    
    for task_id, (L, m) in enumerate(config_list):
        print(f"Generating task {task_id}: L={L}, m={m}")
        
        # Use existing RHM class exactly as-is
        rhm = RandomHierarchyModel(
            num_features=32,        # vocabulary size
            num_classes=10,         # number of classes  
            num_synonyms=m,         # multiplicity
            tuple_size=2,           # size of low-level representations
            num_layers=L,           # number of levels in hierarchy
            seed_rules=task_id,     # different rules per task
            seed_sample=42,         # fixed for reproducibility
            train_size=samples_per_config,
            test_size=0,
            input_format='long',    # gets integer sequences
            replacement=True
        )
        
        # Extract data from RHM object
        sequences = rhm.features   # Shape: [samples_per_config, sequence_length]
        labels = rhm.labels        # Shape: [samples_per_config]
        rules = rhm.rules          # Production rules dictionary
        
        # Convert to lists (HuggingFace prefers lists)
        sequences_list = sequences.tolist()
        labels_list = labels.tolist()
        
        # Store rules for this task
        all_rules[task_id] = rules
        
        # Add to combined dataset
        all_input_ids.extend(sequences_list)
        all_labels.extend(labels_list)
        all_task_ids.extend([task_id] * len(sequences_list))
        all_config_L.extend([L] * len(sequences_list))
        all_config_m.extend([m] * len(sequences_list))
        all_lengths.extend([len(seq) for seq in sequences_list])
    
    # Create HuggingFace Dataset
    dataset_dict = {
        'input_ids': all_input_ids,    # Raw integer sequences
        'labels': all_labels,          # Classification targets
        'task_id': all_task_ids,       # Which RHM configuration  
        'config_L': all_config_L,      # Hierarchy depth
        'config_m': all_config_m,      # Multiplicity
        'length': all_lengths          # Sequence length
    }
    
    dataset = Dataset.from_dict(dataset_dict)
    
    # Save dataset
    os.makedirs(output_dir, exist_ok=True)
    dataset.save_to_disk(f"{output_dir}/mixed_rhm_dataset")
    
    # Save metadata
    metadata = {
        'configs': [{'task_id': i, 'L': L, 'm': m} for i, (L, m) in enumerate(config_list)],
        'vocab_size': 32,
        'num_classes': 10, 
        'tuple_size': 2,
        'samples_per_config': samples_per_config,
        'rules': all_rules
    }
    
    with open(f"{output_dir}/metadata.pkl", 'wb') as f:
        pickle.dump(metadata, f)
    
    print(f"Dataset saved to {output_dir}")
    print(f"Total samples: {len(dataset)}")
    print(f"Sequence length range: {min(dataset['length'])} - {max(dataset['length'])}")
    
    return dataset, metadata

# Usage - Generate mixed RHM dataset
config_list = [(2,4), (2,8), (3,4), (3,8), (3,16)]
dataset, metadata = create_hf_dataset_from_rhm(config_list, samples_per_config=2000)

Generating task 0: L=2, m=4
Generating task 1: L=2, m=8
Generating task 2: L=3, m=4
Generating task 3: L=3, m=8
Generating task 4: L=3, m=16


Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset saved to ./hf_datasets
Total samples: 10000
Sequence length range: 4 - 8


In [ ]:


def create_hf_dataset_from_rhm(config_list, samples_per_config=1000, output_dir='./hf_datasets', 
                               max_length=2048, pack_sequences=True):
    """Create dataset for causal language modeling (next-token prediction) from RHM sequences
    
    Args:
        config_list: List of (L, m) tuples for different RHM configurations
        samples_per_config: Number of samples per configuration
        output_dir: Directory to save the dataset
        max_length: Maximum sequence length for packing
        pack_sequences: Whether to pack sequences for CLM (True) or keep separate for classification (False)
    
    Returns:
        dataset: HuggingFace Dataset
        metadata: Dictionary with dataset metadata
    """
    
    all_sequences = []
    all_labels = []  # Keep labels for analysis even in CLM mode
    all_task_ids = []
    all_config_L = []
    all_config_m = []
    all_lengths = []
    all_rules = {}
    
    print("Generating RHM sequences...")
    
    for task_id, (L, m) in enumerate(config_list):
        print(f"Generating task {task_id}: L={L}, m={m}")
        
        try:
            # Generate RHM data
            rhm = RandomHierarchyModel(
                num_features=32,        # vocabulary size (0-31, +1 shift makes it 1-32)
                num_classes=10,         # number of classes
                num_synonyms=m,         # multiplicity
                tuple_size=2,          # size of low-level representations
                num_layers=L,          # number of levels in hierarchy
                seed_rules=task_id,    # different rules per task
                seed_sample=42,        # fixed for reproducibility
                train_size=samples_per_config,
                test_size=0,
                input_format='long',   # gets integer sequences (1-based)
                replacement=True
            )
            
            # Extract data
            sequences = rhm.features  # Shape: [samples_per_config, sequence_length]
            labels = rhm.labels      # Shape: [samples_per_config]
            rules = rhm.rules        # Production rules dictionary
            
            # Convert to lists
            if hasattr(sequences, 'tolist'):
                sequences_list = sequences.tolist()
            else:
                sequences_list = [list(seq) for seq in sequences]
                
            if hasattr(labels, 'tolist'):
                labels_list = labels.tolist()
            else:
                labels_list = list(labels)
            
            # Store metadata
            all_rules[task_id] = rules
            all_sequences.extend(sequences_list)
            all_labels.extend(labels_list)
            all_task_ids.extend([task_id] * len(sequences_list))
            all_config_L.extend([L] * len(sequences_list))
            all_config_m.extend([m] * len(sequences_list))
            all_lengths.extend([len(seq) for seq in sequences_list])
            
            print(f"  Generated {len(sequences_list)} sequences, avg length: {sum(len(seq) for seq in sequences_list) / len(sequences_list):.1f}")
            
        except Exception as e:
            print(f"Error generating task {task_id}: {e}")
            continue
    
    print(f"\nTotal sequences generated: {len(all_sequences)}")
    print(f"Sequence length range: {min(all_lengths)} - {max(all_lengths)}")
    
    if pack_sequences:
        # Pack sequences for causal language modeling
        print("Packing sequences for causal language modeling...")
        packed_input_ids, packed_metadata = pack_sequences_for_clm(
            all_sequences, 
            all_task_ids, 
            all_config_L, 
            all_config_m, 
            all_labels,
            max_length=max_length
        )
        
        # Create CLM dataset (input_ids and labels are shifted versions)
        dataset_dict = {
            'input_ids': [seq[:-1] for seq in packed_input_ids if len(seq) > 1],  # All but last token
            'labels': [seq[1:] for seq in packed_input_ids if len(seq) > 1],     # All but first token (shifted)
            'length': [len(seq)-1 for seq in packed_input_ids if len(seq) > 1],  # Length of input_ids
            'task_composition': [meta for meta, seq in zip(packed_metadata, packed_input_ids) if len(seq) > 1],  # Which tasks are in each packed sequence
        }
        
        print(f"Packed into {len(dataset_dict['input_ids'])} sequences")
        print(f"Average packed length: {sum(dataset_dict['length']) / len(dataset_dict['length']):.1f}")
        
    else:
        # Keep sequences separate (for classification or analysis)
        print("Keeping sequences separate...")
        dataset_dict = {
            'input_ids': all_sequences,
            'labels': all_labels,
            'task_id': all_task_ids,
            'config_L': all_config_L,
            'config_m': all_config_m,
            'length': all_lengths
        }
    
    # Create HuggingFace Dataset
    dataset = Dataset.from_dict(dataset_dict)
    
    # Save dataset
    os.makedirs(output_dir, exist_ok=True)
    dataset.save_to_disk(f"{output_dir}/rhm_dataset")
    
    # Save metadata
    metadata = {
        'configs': [{'task_id': i, 'L': L, 'm': m} for i, (L, m) in enumerate(config_list)],
        'vocab_size': 32,
        'num_classes': 10,
        'tuple_size': 2,
        'samples_per_config': samples_per_config,
        'max_length': max_length,
        'packed': pack_sequences,
        'rules': all_rules,
        'total_sequences': len(all_sequences),
        'sequence_stats': {
            'min_length': min(all_lengths) if all_lengths else 0,
            'max_length': max(all_lengths) if all_lengths else 0,
            'avg_length': sum(all_lengths) / len(all_lengths) if all_lengths else 0
        }
    }
    
    with open(f"{output_dir}/metadata.pkl", 'wb') as f:
        pickle.dump(metadata, f)
    
    print(f"\nDataset saved to {output_dir}")
    print(f"Dataset format: {'Causal LM (packed)' if pack_sequences else 'Separate sequences'}")
    
    return dataset, metadata



In [ ]:

def pack_sequences_for_clm(sequences: List[List[int]], 
                          task_ids: List[int], 
                          config_L: List[int], 
                          config_m: List[int], 
                          labels: List[int],
                          max_length: int = 2048, 
                          eos_token_id: int = 0) -> Tuple[List[List[int]], List[Dict]]:
    """Pack sequences end-to-end for causal language modeling
    
    Args:
        sequences: List of integer sequences
        task_ids, config_L, config_m, labels: Metadata for each sequence
        max_length: Maximum packed sequence length
        eos_token_id: Token to separate sequences (0 is reserved for padding/separation)
    
    Returns:
        packed_sequences: List of packed integer sequences
        packed_metadata: List of metadata for each packed sequence
    """
    
    packed_sequences = []
    packed_metadata = []
    
    current_sequence = []
    current_metadata = {
        'task_ids': [],
        'config_L': [],
        'config_m': [],
        'labels': [],
        'num_sequences': 0,
        'sequence_boundaries': []  # Track where each original sequence ends
    }
    
    for i, seq in enumerate(sequences):
        # Add EOS token between sequences (except for the very first)
        if current_sequence and eos_token_id is not None:
            seq_with_eos = [eos_token_id] + seq
        else:
            seq_with_eos = seq
        
        # Check if adding this sequence would exceed max_length
        if len(current_sequence) + len(seq_with_eos) <= max_length:
            # Add sequence to current pack
            current_sequence.extend(seq_with_eos)
            current_metadata['task_ids'].append(task_ids[i])
            current_metadata['config_L'].append(config_L[i])
            current_metadata['config_m'].append(config_m[i])
            current_metadata['labels'].append(labels[i])
            current_metadata['num_sequences'] += 1
            current_metadata['sequence_boundaries'].append(len(current_sequence))
            
        else:
            # Save current packed sequence if it has content
            if current_sequence:
                packed_sequences.append(current_sequence)
                packed_metadata.append(current_metadata)
            
            # Start new packed sequence
            current_sequence = seq_with_eos[:max_length]  # Truncate if single sequence too long
            current_metadata = {
                'task_ids': [task_ids[i]],
                'config_L': [config_L[i]],
                'config_m': [config_m[i]],
                'labels': [labels[i]],
                'num_sequences': 1,
                'sequence_boundaries': [len(current_sequence)]
            }
    
    # Don't forget the last packed sequence
    if current_sequence:
        packed_sequences.append(current_sequence)
        packed_metadata.append(current_metadata)
    
    return packed_sequences, packed_metadata


def create_clm_dataloader(dataset, batch_size=8, pad_token_id=0):
    """Create DataLoader for causal language modeling with dynamic padding
    
    Args:
        dataset: HuggingFace Dataset with 'input_ids' and 'labels'
        batch_size: Batch size
        pad_token_id: Token ID for padding (0 by default)
    
    Returns:
        DataLoader with properly padded batches
    """
    
    def collate_fn(batch):
        # Extract input_ids and labels
        input_ids = [torch.tensor(item['input_ids']) for item in batch]
        labels = [torch.tensor(item['labels']) for item in batch]
        
        # Pad sequences to same length in batch
        from torch.nn.utils.rnn import pad_sequence
        input_ids_padded = pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)
        labels_padded = pad_sequence(labels, batch_first=True, padding_value=-100)  # -100 is ignored in loss
        
        # Create attention mask (1 for real tokens, 0 for padding)
        attention_mask = (input_ids_padded != pad_token_id).long()
        
        return {
            'input_ids': input_ids_padded,
            'attention_mask': attention_mask,
            'labels': labels_padded
        }
    
    from torch.utils.data import DataLoader
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)


# Example usage:
if __name__ == "__main__":
    # Define different RHM configurations
    config_list = [
        (2, 2),  # L=2, m=2 (shallow, low multiplicity)
        (3, 2),  # L=3, m=2 (deeper)
        (2, 4),  # L=2, m=4 (shallow, high multiplicity)
        (4, 3),  # L=4, m=3 (deep, medium multiplicity)
    ]
    
    # Create dataset for causal language modeling
    dataset, metadata = create_hf_dataset_from_rhm(
        config_list=config_list,
        samples_per_config=1000,
        output_dir='./rhm_clm_dataset',
        max_length=2048,
        pack_sequences=True  # Set to False for classification tasks
    )
    
    print(f"\nDataset created with {len(dataset)} packed sequences")
    print(f"Vocabulary size: 1-32 (0 reserved for EOS/padding)")
    print(f"Sample input_ids length: {len(dataset[0]['input_ids'])}")
    print(f"Sample labels length: {len(dataset[0]['labels'])}")
    
    # Create DataLoader
    dataloader = create_clm_dataloader(dataset, batch_size=4)
    
    # Test batch
    for batch in dataloader:
        print(f"\nBatch shapes:")
        print(f"  input_ids: {batch['input_ids'].shape}")
        print(f"  attention_mask: {batch['attention_mask'].shape}")
        print(f"  labels: {batch['labels'].shape}")
        break


hook

# test training dataset


In [1]:
from datasets import load_from_disk

# Load the dataset we just created
dataset = load_from_disk('./hf_datasets/mixed_rhm_dataset')

/Users/jliu/anaconda3/envs/genai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from transformers import PreTrainedTokenizerFast

# Create simple tokenizer for our integer sequences
def create_rhm_tokenizer(vocab_size=32):
    # Vocabulary: 0 is reserved for padding, 1-32 are actual tokens
    vocab = {str(i): i for i in range(vocab_size + 1)}
    
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=None,
        vocab=vocab,
        pad_token="0",
        unk_token="0",
    )
    return tokenizer



# Preprocessing function
def preprocess_function(examples):
    # The input_ids are already tokenized integers
    # Just need to handle padding (HuggingFace will do this automatically)
    return {
        'input_ids': examples['input_ids'],
        'labels': examples['labels']
    }



In [4]:
tokenizer = create_rhm_tokenizer()

ValueError: Converting from Tiktoken failed, if a converter for SentencePiece is available, provide a model path with a SentencePiece tokenizer.model file.Currently available slow->fast convertors: ['AlbertTokenizer', 'BartTokenizer', 'BarthezTokenizer', 'BertTokenizer', 'BigBirdTokenizer', 'BlenderbotTokenizer', 'CamembertTokenizer', 'CLIPTokenizer', 'CodeGenTokenizer', 'ConvBertTokenizer', 'DebertaTokenizer', 'DebertaV2Tokenizer', 'DistilBertTokenizer', 'DPRReaderTokenizer', 'DPRQuestionEncoderTokenizer', 'DPRContextEncoderTokenizer', 'ElectraTokenizer', 'FNetTokenizer', 'FunnelTokenizer', 'GPT2Tokenizer', 'HerbertTokenizer', 'LayoutLMTokenizer', 'LayoutLMv2Tokenizer', 'LayoutLMv3Tokenizer', 'LayoutXLMTokenizer', 'LongformerTokenizer', 'LEDTokenizer', 'LxmertTokenizer', 'MarkupLMTokenizer', 'MBartTokenizer', 'MBart50Tokenizer', 'MPNetTokenizer', 'MobileBertTokenizer', 'MvpTokenizer', 'NllbTokenizer', 'OpenAIGPTTokenizer', 'PegasusTokenizer', 'Qwen2Tokenizer', 'RealmTokenizer', 'Refor

In [ ]:
# Apply preprocessing
processed_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=['task_id', 'config_L', 'config_m', 'length']  # Keep only model inputs
)